# Timeseries Tracking Data with GEFF

This notebook shows how to generate temporally coherent mother-machine training data for tracking models, with lineage exported in GEFF format.


## What this workflow does

- Renders sequential frames (no random frame shuffling).
- Keeps optical/noise parameters constant within each series.
- Produces multiple variants (`n_series`) from one simulation.
- Exports lineage for each series as `lineage.geff`.


In [ ]:
%load_ext autoreload
%autoreload 2

from SyMBac.config_models import (
    BrownianJitterSpec,
    DatasetOutputConfig,
    RenderConfig,
    SimulationCellSpec,
    SimulationGeometrySpec,
    SimulationLowLevelSpec,
    SimulationPhysicsSpec,
    SimulationRuntimeSpec,
    SimulationSpec,
    TimeseriesDatasetPlan,
)
from SyMBac.simulation import Simulation
from SyMBac.PSF import PSF_generator, Camera
from SyMBac.renderer import Renderer
from SyMBac.misc import get_sample_images
import os
os.environ['DISPLAY'] = ':1'


## 1. Run a mother-machine simulation

You can tune low-level physical bendiness with `cell_config_overrides` and
`physics_config_overrides` directly from the notebook.


In [ ]:
real_image = get_sample_images()["E. coli 100x"]

# Optional low-level overrides for bendiness/rigidity.
cell_config_overrides = {
    "MAX_BEND_ANGLE": 0.01,            # higher => more bend allowed
    "STIFFNESS": 120_000.0,            # lower => softer bend constraint
    "PIVOT_JOINT_STIFFNESS": 3_000.0,  # lower => more flexible joints
    "NOISE_STRENGTH": 0.08,            # higher => more wiggle
}

physics_config_overrides = {
    "ITERATIONS": 120,
    "DAMPING": 0.5,
}

sim_spec = SimulationSpec(
    geometry=SimulationGeometrySpec(
        trench_length=14.25,
        trench_width=1.45,
        pix_mic_conv=0.065,
        resize_amount=3,
    ),
    cell=SimulationCellSpec(
        cell_max_length=6.65,
        cell_width=1.1,
        max_length_std=0.0,
        width_std=0.0,
        lysis_p=0.0,
    ),
    physics=SimulationPhysicsSpec(gravity=0.0, phys_iters=15),
    runtime=SimulationRuntimeSpec(
        sim_length=100,
        save_dir="/tmp/symbac_tracking_sim/",
        substeps=100,
    ),
    low_level=SimulationLowLevelSpec(
        cell_config_overrides=cell_config_overrides,
        physics_config_overrides=physics_config_overrides,
    ),
    brownian=BrownianJitterSpec(
        application_mode="velocity",
        projection_angular_damping=0.30,
        longitudinal_std=0.004,
        transverse_std=0.002,
        rotation_std=0.002,
        persistence=0.80,
        max_dx_fraction_of_trench_width=0.20,
        max_dy_fraction_of_segment_radius=0.75,
        max_dy_px_floor=1.0,
        max_dtheta=0.03,
        backoff_attempts=5,
    ),
)

my_simulation = Simulation(sim_spec)
my_simulation.run(show_window=True)
my_simulation.draw_opl(do_transformation=False, label_masks=True)


In [ ]:
my_simulation.visualise_in_napari()

## 2. Build optics + renderer


In [ ]:
# A phase contrast kernel
my_kernel = PSF_generator(
    radius = 50, 
    wavelength = 0.75, 
    NA = 1.2, 
    n = 1.3, 
    resize_amount = 3, 
    pix_mic_conv = 0.065, 
    apo_sigma = 20, 
    mode="phase contrast", 
    condenser = "Ph3")
my_kernel.calculate_PSF()
my_kernel.plot_PSF()

my_camera = Camera(baseline=100, sensitivity=2.9, dark_noise=8)

my_renderer = Renderer(
    simulation=my_simulation,
    PSF=my_kernel,
    real_image=real_image,
    camera=my_camera,
    additional_real_images=[real_image],
)


## 3. Set or optimise rendering parameters

If you already have optimised parameters, apply them here. Otherwise run your usual optimisation flow first (manual `RenderTuner`).


In [ ]:
# Create an interactive manual tuner and get a typed RenderConfig back.
my_renderer.select_intensity_napari()
tuner = my_renderer.create_tuner(base_config=RenderConfig(), manual_update=False)
tuner.widget


## 4. Generate one coherent timeseries + GEFF


In [ ]:
single_plan = TimeseriesDatasetPlan(
    burn_in=40,
    sample_amount=0.02,
    n_series=1,
    frames_per_series=200,
)
single_output = DatasetOutputConfig(
    save_dir="/tmp/symbac_tracking_data_single_1/",
    image_format="png",
    mask_dtype="uint16",
    export_geff=True,
    n_jobs=1,
)
single_meta = my_renderer.export_dataset(
    plan=single_plan,
    output=single_output,
    base_config=tuner.current_config(),
    seed=42,
)
single_meta


## 5. Generate multiple variants from the same simulation

This gives multiple independent coherent series (`series_000`, `series_001`, ...) with different fixed optical/noise settings per series.


In [ ]:
multi_plan = TimeseriesDatasetPlan(
    burn_in=40,
    sample_amount=0.10,
    n_series=5,
    frames_per_series=150,
)
multi_output = DatasetOutputConfig(
    save_dir="/tmp/symbac_tracking_data_multi_variant/",
    image_format="tiff",
    mask_dtype="uint16",
    export_geff=True,
    n_jobs=1,
)
multi_variant_meta = my_renderer.export_dataset(
    plan=multi_plan,
    output=multi_output,
    base_config=tuner.current_config(),
    seed=123,
)
multi_variant_meta["n_series"], multi_variant_meta["series"][0]


## 6. Multiple simulations (biological variation) + per-simulation variants

Use an outer loop over simulation parameters, and an inner call to `export_dataset(TimeseriesDatasetPlan(...), ...)` for optical/noise variants.


In [ ]:
simulation_configs = [
    dict(cell_max_length=6.2, cell_width=1.0),
    dict(cell_max_length=6.8, cell_width=1.1),
]

all_runs = []
for sim_idx, cfg in enumerate(simulation_configs):
    spec = SimulationSpec(
        geometry=SimulationGeometrySpec(
            trench_length=14.25,
            trench_width=1.45,
            pix_mic_conv=0.065,
            resize_amount=3,
        ),
        cell=SimulationCellSpec(
            cell_max_length=cfg["cell_max_length"],
            cell_width=cfg["cell_width"],
            max_length_std=0.05,
            width_std=0.02,
            lysis_p=0.01,
        ),
        physics=SimulationPhysicsSpec(gravity=0.0, phys_iters=15),
        runtime=SimulationRuntimeSpec(
            sim_length=400,
            save_dir=f"/tmp/symbac_tracking_sim_{sim_idx:03d}/",
            substeps=100,
        ),
    )
    sim = Simulation(spec)
    sim.run(show_window=False)
    sim.draw_opl(do_transformation=False, label_masks=True)

    renderer = Renderer(simulation=sim, PSF=my_kernel, real_image=real_image, camera=my_camera)
    render_config = tuner.current_config()

    plan = TimeseriesDatasetPlan(
        burn_in=40,
        sample_amount=0.08,
        n_series=3,
        frames_per_series=120,
    )
    output = DatasetOutputConfig(
        save_dir=f"/tmp/symbac_tracking_dataset/sim_{sim_idx:03d}/",
        image_format="tiff",
        mask_dtype="uint16",
        export_geff=True,
        n_jobs=1,
    )
    meta = renderer.export_dataset(
        plan=plan,
        output=output,
        base_config=render_config,
        seed=1000 + sim_idx,
    )
    all_runs.append(meta)

len(all_runs)


## 7. Inspect exported GEFF


In [ ]:
# Requires: pip install "SyMBac[geff]"
# import geff
# graph, metadata = geff.read("/tmp/symbac_tracking_data_single/series_000/lineage.geff")
# print(graph.number_of_nodes(), graph.number_of_edges())
